# Corrected RFI subset with a group split

This notebook replaces neither the old dataset notebook nor `split_indices.npz`. It creates a new local artifact set in which every selected `segment_index` belongs to exactly one split. The output is the required starting point for the corrected 1D-CNN, statistical-model and normalization experiments.

The fitting universe contains only `NBRFI` and `None`. `NoneWNBRFI` is saved separately as hard-negative material and must not enter training, validation-threshold selection or ordinary test metrics.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError('Start Jupyter from the rfimt repository root.')
sys.path.insert(0, str(REPO_ROOT / 'src'))

from rfimt.io import read_metadata
from rfimt.splits import (
    assert_no_group_overlap,
    make_group_split_indices,
    sample_balanced_within_indices,
    summarize_group_splits,
)

## Local inputs and output

The full source artifacts are on the server under `DATA_ROOT`. The notebook writes a new directory there instead of overwriting the historical subset.

The target `8,000 / 1,000 / 1,000` rows per class preserves the old total of 20,000 fitting examples while making the split allocation explicit. Reduce those values only when the group-level audit shows that a class is unavailable in a split.

In [ ]:
DATA_ROOT = Path('/hercules/results/akazantsev/rfim_dataset')
METADATA_PATH = DATA_ROOT / 'B0531+21_59000_48386_channels_meta.csv'
ARRAY_PATH = DATA_ROOT / 'B0531+21_59000_48386_channels.npy'
OUTPUT_DIR = DATA_ROOT / 'B0531+21_59000_48386_group_split_v1'

GROUP_COLUMN = 'segment_index'
LABEL_COLUMN = 'label'
TRAINING_LABELS = ('NBRFI', 'None')
HARD_NEGATIVE_LABEL = 'NoneWNBRFI'
SEED = 42
ROWS_PER_CLASS = {'train': 8_000, 'val': 1_000, 'test': 1_000}

REQUIRED_COLUMNS = {
    'sample_index', 'global_index', 'segment_index', 'channel_index',
    'frequency', 'label', 'original_segment_label',
}

In [ ]:
full_meta = read_metadata(METADATA_PATH)
missing_columns = REQUIRED_COLUMNS.difference(full_meta.columns)
if missing_columns:
    raise ValueError(f'Full metadata is missing required columns: {sorted(missing_columns)}')

full_array = np.load(ARRAY_PATH, mmap_mode='r')
if len(full_meta) != len(full_array):
    raise ValueError(f'Metadata rows ({len(full_meta)}) and array rows ({len(full_array)}) differ.')

candidate_meta = full_meta.loc[full_meta[LABEL_COLUMN].isin(TRAINING_LABELS)].copy()
candidate_meta['source_row_index'] = candidate_meta.index
candidate_meta = candidate_meta.reset_index(drop=True)

hard_negative_meta = full_meta.loc[full_meta[LABEL_COLUMN].eq(HARD_NEGATIVE_LABEL)].copy()
hard_negative_meta['source_row_index'] = hard_negative_meta.index

print('Full rows:', len(full_meta))
print('Candidate label counts:')
display(candidate_meta[LABEL_COLUMN].value_counts())
print('Hard-negative rows:', len(hard_negative_meta))
print('Candidate groups:', candidate_meta[GROUP_COLUMN].nunique())

In [ ]:
candidate_splits = make_group_split_indices(
    candidate_meta,
    group_col=GROUP_COLUMN,
    test_size=0.10,
    val_size=0.10,
    random_state=SEED,
)
assert_no_group_overlap(candidate_meta, candidate_splits, group_col=GROUP_COLUMN)

sampled_candidate_indices = {
    split_name: sample_balanced_within_indices(
        candidate_meta,
        split_indices,
        label_col=LABEL_COLUMN,
        labels_to_sample=TRAINING_LABELS,
        n_per_class=ROWS_PER_CLASS[split_name],
        random_state=SEED + offset,
    )
    for offset, (split_name, split_indices) in enumerate(candidate_splits.items())
}

sampled_candidate_indices = {
    name: np.asarray(indices, dtype=int)
    for name, indices in sampled_candidate_indices.items()
}
sampled_original_indices = {
    name: candidate_meta.iloc[indices]['source_row_index'].to_numpy(dtype=int)
    for name, indices in sampled_candidate_indices.items()
}

selected_candidate_indices = np.concatenate([
    sampled_candidate_indices['train'],
    sampled_candidate_indices['val'],
    sampled_candidate_indices['test'],
])
subset_meta = candidate_meta.iloc[selected_candidate_indices].reset_index(drop=True)
subset_array = np.asarray(full_array[np.concatenate([
    sampled_original_indices['train'],
    sampled_original_indices['val'],
    sampled_original_indices['test'],
])])

n_train = len(sampled_candidate_indices['train'])
n_val = len(sampled_candidate_indices['val'])
subset_splits = {
    'train': np.arange(0, n_train),
    'val': np.arange(n_train, n_train + n_val),
    'test': np.arange(n_train + n_val, len(subset_meta)),
}
assert_no_group_overlap(subset_meta, subset_splits, group_col=GROUP_COLUMN)
split_summary = summarize_group_splits(
    subset_meta, subset_splits, group_col=GROUP_COLUMN, label_col=LABEL_COLUMN
)
display(split_summary)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
subset_meta.to_csv(OUTPUT_DIR / 'subset_channels_meta.csv', index=False)
np.save(OUTPUT_DIR / 'subset_channels.npy', subset_array)
np.savez(OUTPUT_DIR / 'group_split_indices.npz', **subset_splits)
split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv', index=False)
hard_negative_meta.to_csv(OUTPUT_DIR / 'hard_negative_meta.csv', index=False)
np.save(
    OUTPUT_DIR / 'hard_negative_source_row_indices.npy',
    hard_negative_meta['source_row_index'].to_numpy(dtype=int),
)

provenance = {
    'dataset_id': 'B0531+21_59000_48386_group_split_v1',
    'source_metadata': str(METADATA_PATH),
    'source_array': str(ARRAY_PATH),
    'group_column': GROUP_COLUMN,
    'seed': SEED,
    'training_labels': list(TRAINING_LABELS),
    'hard_negative_label': HARD_NEGATIVE_LABEL,
    'requested_rows_per_class': ROWS_PER_CLASS,
    'actual_subset_rows': int(len(subset_meta)),
    'actual_split_rows': {name: int(len(indices)) for name, indices in subset_splits.items()},
    'actual_split_groups': {
        name: int(subset_meta.iloc[indices][GROUP_COLUMN].nunique())
        for name, indices in subset_splits.items()
    },
}
with (OUTPUT_DIR / 'dataset_provenance.json').open('w', encoding='utf-8') as handle:
    json.dump(provenance, handle, indent=2, sort_keys=True)
    handle.write('\n')

print(f'Wrote corrected local artifacts to: {OUTPUT_DIR}')

## Acceptance check

Do not proceed to model training until the displayed summary shows all three splits, both fitting labels where expected, and the preceding overlap assertions have passed. Preserve this output and `dataset_provenance.json`; the later experiment configuration must reference this artifact directory.

`NoneWNBRFI` is exported only as metadata plus source-row indices. A later evaluation notebook may load those rows from the immutable full array, but must identify that metric as hard-negative evaluation rather than ordinary test performance.